# Day 28 — FAANG / Top Product Company Interview Practice (Week 4 · Day 7)

**18 questions · 8 categories · Tags: [HARD] × 8 · [FAANG] × 10**

| # | Category | Topic | Tag |
| --- | --- | --- | --- |
| 1 | SQL Advanced | Sweep-line maximum concurrency | [FAANG] |
| 2 | SQL Advanced | RFM segmentation with NTILE quintiles | [HARD] |
| 3 | Python | CRDTs: PN-counter & LWW set | [FAANG] |
| 4 | Python | Skip list from scratch | [FAANG] |
| 5 | PySpark | Incremental AI-function enrichment (GenAI pipeline) | [FAANG] |
| 6 | PySpark | Higher-order functions on nested arrays | [HARD] |
| 7 | Spark Optimization | Dynamic resource allocation policy | [HARD] |
| 8 | Spark Optimization | Lineage, caching & checkpoint truncation | [HARD] |
| 9 | ETL | Debezium-style log-based CDC | [FAANG] |
| 10 | ETL | Deterministic HMAC PII tokenization | [HARD] |
| 11 | ETL | Metadata-driven ingestion framework | [HARD] |
| 12 | Data Modeling | Ratio & non-additive metrics | [HARD] |
| 13 | Data Modeling | SaaS recurring-billing & MRR waterfall | [FAANG] |
| 14 | Snowflake | ACCESS_HISTORY governance forensics | [FAANG] |
| 15 | Snowflake | Semi-structured VARIANT at scale | [HARD] |
| 16 | Monitoring | Data-product SLA contracts | [FAANG] |
| 17 | Monitoring | Test data management (masked clones + synthetic) | [FAANG] |
| 18 | Monitoring | FinOps cost-anomaly attribution (live) | [FAANG] |

Every solution cell is self-contained, runnable on serverless compute, and ends with assertions that verify the result. Run top to bottom. Score yourself at the end.

In [0]:
# ---- Setup: unique scratch paths so repeated runs never collide ----
import uuid
from pyspark.sql import functions as F
RUN = uuid.uuid4().hex[:8]
BASE = f"/Volumes/workspace/default/etl_test_volume/d28_{RUN}"
print("scratch base:", BASE)

## Q1 · SQL Advanced — Sweep-Line Maximum Concurrency [FAANG]

**Scenario (Uber/DoorDash style):** You have a table of user sessions with `start_ts` and `end_ts`. A dashboard needs the **peak number of concurrent sessions** and the **exact timestamp of the peak**, plus the full concurrency profile over time. Sessions can start while others are running; intervals overlap arbitrarily.

**Requirements:**
1. Solve it with a single SQL query (one pass over the data, no self-join per session).
2. Return `peak_concurrency` and the timestamp it occurs (`peak_at`).
3. Also produce the full time series `(ts, concurrent_sessions)` for a chart.

**Interview traps:** the O(n²) self-join (`JOIN t2 ON t2.start < t1.start AND t2.end > t1.start`) doesn't scale; and boundary handling — if one session ends exactly when another starts, do they overlap? Decide and defend your tie-break rule.

In [0]:
import datetime as dt
fmt = "%Y-%m-%d %H:%M"
sessions = [
    ("s1", "2026-11-19 09:00", "2026-11-19 09:30"),
    ("s2", "2026-11-19 09:10", "2026-11-19 09:20"),
    ("s3", "2026-11-19 09:15", "2026-11-19 09:45"),
    ("s4", "2026-11-19 09:40", "2026-11-19 10:10"),
    ("s5", "2026-11-19 10:05", "2026-11-19 10:20"),
]
# pass REAL datetime objects for TIMESTAMP columns (strings -> createDataFrame type trap)
rows = [(s, dt.datetime.strptime(a, fmt), dt.datetime.strptime(b, fmt)) for s, a, b in sessions]
df = spark.createDataFrame(rows, "sid STRING, start_ts TIMESTAMP, end_ts TIMESTAMP")
df.createOrReplaceTempView("q1_sessions")

# SOLUTION — sweep line: +1 at every start, -1 at every end, running sum = concurrency.
# Tie-break: ORDER BY (ts, delta) puts -1 BEFORE +1 at the same instant,
# i.e. an interval ending exactly when another starts is NOT overlapping (half-open [start, end)).
peak_sql = """
WITH events AS (
  SELECT start_ts AS ts,  1 AS delta FROM q1_sessions
  UNION ALL
  SELECT end_ts   AS ts, -1 AS delta FROM q1_sessions
),
ordered AS (
  SELECT ts, delta,
         SUM(delta) OVER (ORDER BY ts, delta
                           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running
  FROM events
)
SELECT MAX(running) AS peak_concurrency, MAX_BY(ts, running) AS peak_at
FROM ordered
"""
res = spark.sql(peak_sql).collect()
peak, peak_at = res[0]["peak_concurrency"], res[0]["peak_at"]
print("sweep-line peak:", peak, "at", peak_at)

# full profile for the chart
prof = spark.sql("""
WITH events AS (
  SELECT start_ts AS ts,  1 AS delta FROM q1_sessions
  UNION ALL
  SELECT end_ts   AS ts, -1 AS delta FROM q1_sessions
),
ordered AS (
  SELECT ts, delta,
         SUM(delta) OVER (ORDER BY ts, delta
                           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running
  FROM events
)
SELECT ts, running AS concurrent_sessions FROM ordered ORDER BY ts
""")
prof.show(12, truncate=False)

# VERIFY — brute-force on a 1-minute grid must agree with the sweep line
t = dt.datetime(2026, 11, 19, 9, 0)
brute_peak, brute_at = 0, None
while t <= dt.datetime(2026, 11, 19, 10, 20):
    n = sum(1 for _, a, b in rows if a <= t < b)   # half-open: same rule as the SQL tie-break
    if n > brute_peak:
        brute_peak, brute_at = n, t
    t += dt.timedelta(minutes=1)
assert peak == brute_peak, (peak, brute_peak)
assert peak_at.strftime("%H:%M") == brute_at.strftime("%H:%M")
assert prof.count() == 2 * len(sessions)
print("Q1 OK — sweep-line == brute force:", peak, "concurrent at", peak_at)
print("Talk track: O(n log n) via window sort; MAX_BY for argmax; half-open vs closed intervals;")
print("a grid-based brute force is O(n * range/step) and breaks on irregular timestamps.")

## Q2 · SQL Advanced — RFM Segmentation with NTILE Quintiles [HARD]

**Scenario (Amazon/Flipkart style):** Marketing wants a classic **RFM (Recency · Frequency · Monetary)** segmentation of customers from an orders fact table, computed **entirely in SQL**, for a nightly campaign table.

**Requirements:**
1. Per customer: days since last order (as of a fixed snapshot date), distinct order days, total spend.
2. Score each dimension into quintiles 1–5 with `NTILE(5)` — best customers get 5 on all three (note: recency must be inverted, frequency/monetary ascending).
3. Map quintile triples to named segments (Champions, New/promising, Loyal at risk, Hibernating, Core).
4. Aggregate segment revenue and prove it reconciles to total order value.

**Interview traps:** averaging pre-computed ratios (Q12's disease) — here you must reason about **which direction each quintile sorts** and why `NTILE` beats `PERCENT_RANK` for equal-sized buckets.

In [0]:
import datetime as dt
from collections import Counter
orders = [
    ("u1", "2026-11-18", 250.0), ("u1", "2026-11-01", 100.0), ("u1", "2026-09-20", 75.0),
    ("u2", "2026-02-10", 500.0),
    ("u3", "2026-11-17", 40.0), ("u3", "2026-11-10", 60.0), ("u3", "2026-10-30", 90.0), ("u3", "2026-10-01", 120.0),
    ("u4", "2026-11-16", 1000.0), ("u4", "2026-11-02", 800.0),
    ("u5", "2026-03-05", 20.0), ("u5", "2026-03-04", 10.0),
    ("u6", "2026-11-15", 300.0), ("u6", "2026-06-15", 200.0), ("u6", "2026-01-15", 100.0),
    ("u7", "2026-11-19", 600.0), ("u7", "2026-11-18", 400.0), ("u7", "2026-11-17", 200.0), ("u7", "2026-11-16", 800.0),
    ("u8", "2026-04-01", 700.0),
    ("u9", "2026-11-05", 150.0), ("u9", "2026-10-05", 250.0), ("u9", "2026-08-05", 350.0),
    ("u10", "2026-11-19", 25.0),
]
SNAP = dt.date(2026, 11, 19)
rows = [(u, dt.date.fromisoformat(d), float(a)) for u, d, a in orders]   # real dates, not strings
spark.createDataFrame(rows, "uid STRING, order_date DATE, amount DOUBLE").createOrReplaceTempView("q2_orders")

rfm = spark.sql("""
WITH per_user AS (
  SELECT uid,
         DATEDIFF(CAST('2026-11-19' AS DATE), MAX(order_date)) AS recency_days,
         COUNT(DISTINCT order_date) AS frequency,
         ROUND(SUM(amount), 2) AS monetary
  FROM q2_orders GROUP BY uid
),
scored AS (
  SELECT *,
         NTILE(5) OVER (ORDER BY recency_days DESC) AS r_quintile,  -- smallest recency = best = 5
         NTILE(5) OVER (ORDER BY frequency ASC)    AS f_quintile,
         NTILE(5) OVER (ORDER BY monetary ASC)     AS m_quintile
  FROM per_user
)
SELECT *,
       CASE WHEN r_quintile >= 4 AND f_quintile >= 4 AND m_quintile >= 4 THEN 'Champions'
            WHEN r_quintile >= 4 AND f_quintile <= 2                    THEN 'New / promising'
            WHEN r_quintile <= 2 AND f_quintile >= 4 AND m_quintile >= 4 THEN 'Loyal at risk'
            WHEN r_quintile <= 2 AND f_quintile <= 2 AND m_quintile <= 2 THEN 'Hibernating'
            ELSE 'Core / needs nurture' END AS segment
FROM scored
ORDER BY uid
""")
rfm.createOrReplaceTempView("q2_rfm")
rfm.show(20, truncate=False)
got = {r["uid"]: r for r in rfm.collect()}

# VERIFY 1 — SQL RFM values == python reference computed from the same rows
rec = {}
for u, d, a in rows:
    e = rec.setdefault(u, {"last": d, "freq": set(), "mon": 0.0})
    e["last"] = max(e["last"], d); e["freq"].add(d); e["mon"] += a
for u, e in rec.items():
    assert got[u]["recency_days"] == (SNAP - e["last"]).days
    assert got[u]["frequency"] == len(e["freq"])
    assert abs(got[u]["monetary"] - round(e["mon"], 2)) < 0.01

# VERIFY 2 — labels follow the rules exactly (never trust a hand-computed label)
def seg_of(r, f, m):
    if r >= 4 and f >= 4 and m >= 4: return "Champions"
    if r >= 4 and f <= 2: return "New / promising"
    if r <= 2 and f >= 4 and m >= 4: return "Loyal at risk"
    if r <= 2 and f <= 2 and m <= 2: return "Hibernating"
    return "Core / needs nurture"
for u, row in got.items():
    assert row["segment"] == seg_of(row["r_quintile"], row["f_quintile"], row["m_quintile"])

# VERIFY 3 — equal quintiles (10 users / 5 tiles = exactly 2 each) + revenue reconciliation
for q in ("r_quintile", "f_quintile", "m_quintile"):
    c = Counter(row[q] for row in got.values())
    assert set(c.values()) == {2}, (q, c)
seg = spark.sql("SELECT segment, COUNT(*) AS users, ROUND(SUM(monetary),2) AS revenue FROM q2_rfm GROUP BY segment")
total = float(seg.agg(F.sum("revenue")).collect()[0][0])
assert abs(total - round(sum(a for _, _, a in orders), 2)) < 0.01
seg.orderBy(F.desc("revenue")).show(truncate=False)
print("Q2 OK — RFM verified: values match reference, labels follow rules, quintiles equal-sized, revenue reconciles")
print("Talk track: NTILE forces equal bucket sizes even with ties; PERCENT_RANK does not.")
print("Recency sorts DESC so 5 = most recent. Frequency/monetary ASC so 5 = best. Say this out loud.")

## Q3 · Python — CRDTs: PN-Counter & LWW Set [FAANG]

**Scenario (Google Docs/Figma offline-first style):** Two clients edit the same data while offline. When they reconnect, their replicas must merge **without a coordinator and without conflicts** — exactly what CRDTs (Conflict-free Replicated Data Types) guarantee: merge is commutative, associative, and idempotent, so all replicas converge no matter the merge order.

**Requirements:**
1. Implement a **PN-Counter** (grow/shrink counter): two G-counters (increments P, decrements N); merge = element-wise `max`; value = `P - N`.
2. Implement an **LWW-Set** (last-write-wins set): per element keep the max timestamp for `add` and for `remove`; an element is present iff its add-timestamp beats its remove-timestamp.
3. Prove the CRDT properties: convergence after divergent edits, merge order-independence, and idempotent double-merge.

**Interview traps:** why merge uses `max`, not `sum` (duplicate deliveries must not double-count); why wall-clock timestamps make LWW only probabilistically correct (clock skew) while a monotonic/logical clock per replica is safe.

In [0]:
import random
random.seed(28)

class GCounter:
    """Grow-only counter: one slot per replica. merge = element-wise max."""
    def __init__(self, n_replicas):
        self.c = [0] * n_replicas
    def incr(self, replica, by=1):
        self.c[replica] += by
        return self
    def value(self):
        return sum(self.c)
    def merge(self, other):
        self.c = [max(a, b) for a, b in zip(self.c, other.c)]
        return self

class PNCounter:
    """PN-counter = GCounter(P) - GCounter(N): supports incr AND decr safely."""
    def __init__(self, n_replicas):
        self.p, self.n = GCounter(n_replicas), GCounter(n_replicas)
    def incr(self, replica, by=1): self.p.incr(replica, by); return self
    def decr(self, replica, by=1): self.n.incr(replica, by); return self
    def value(self): return self.p.value() - self.n.value()
    def merge(self, other):
        self.p.merge(other.p); self.n.merge(other.n); return self

class LWWSet:
    """Last-write-wins set: present iff add_ts > remove_ts (max wins on merge)."""
    def __init__(self):
        self.adds, self.removes = {}, {}          # element -> max timestamp seen
    def add(self, e, ts):
        self.adds[e] = max(self.adds.get(e, 0), ts)
        return self
    def remove(self, e, ts):
        if e in self.adds:                        # remember intent even if it races the add
            self.removes[e] = max(self.removes.get(e, 0), ts)
        return self
    def __contains__(self, e):
        return self.adds.get(e, 0) > self.removes.get(e, 0)
    def elements(self):
        return sorted(e for e in self.adds if e in self)
    def merge(self, other):
        for d_self, d_other in ((self.adds, other.adds), (self.removes, other.removes)):
            for k, v in d_other.items():
                d_self[k] = max(d_self.get(k, 0), v)
        return self

# ---- PN-COUNTER: replica0 +5/-2 (=3), replica1 +3/-4 (=-1); merged = (5+3)-(2+4) = 2 ----
a, b = PNCounter(2), PNCounter(2)
a.incr(0, 5).decr(0, 2)
b.incr(1, 3).decr(1, 4)
assert a.value() == 3 and b.value() == -1
m1 = PNCounter(2); m1.merge(a); m1.merge(b)
m2 = PNCounter(2); m2.merge(b); m2.merge(a)     # different merge ORDER
assert m1.value() == m2.value() == 2            # converge either way
m1.merge(a)                                      # duplicate delivery
assert m1.value() == 2                          # idempotent
print("Q3a OK — PN-Counter: local 3 and -1 -> merged", m1.value(), "; order-independent & idempotent")

# ---- LWW-SET: concurrent adds, remove, merge both ways, duplicate merge, re-add ----
s1, s2 = LWWSet(), LWWSet()
s1.add("doc:home", 10); s1.add("doc:taxes", 12)
s2.add("doc:about", 8); s2.add("doc:home", 15)  # newer add for doc:home from replica 2
assert "doc:home" in s1 and "doc:about" in s2
s1.remove("doc:taxes", 20)
assert "doc:taxes" not in s1
s1.merge(s2)
assert "doc:home" in s1                          # the newer add (15) survived
assert "doc:taxes" not in s1 and "doc:about" in s1
s2.merge(s1)                                      # merge back the other way
assert s1.elements() == s2.elements() == ["doc:about", "doc:home"]
s2.merge(s1)                                      # duplicate merge
assert s2.elements() == ["doc:about", "doc:home"]  # still idempotent
s1.add("doc:taxes", 25)                           # re-add AFTER a remove -> resurrects
assert "doc:taxes" in s1
print("Q3b OK — LWW-Set: converged; duplicate merge idempotent; re-add resurrects")
print("Q3 OK — CRDTs: PN-Counter + LWW-Set with convergence, commutativity, idempotence proven")
print("Talk track: max() not sum() because delivery can duplicate; monotonic clocks/HLCs beat wall time;")
print("this is the Dynamo/Cassandra counter lineage, and Riak's actual data model.")

## Q4 · Python — Skip List from Scratch [FAANG]

**Scenario (Redis style):** Redis's sorted set (`ZADD`/`ZRANGE`) is backed by a **skip list** — an ordered structure with O(log n) expected insert/search/delete, no rebalancing, and cheap range scans. Interviewers make you build one to prove you understand probabilistic balancing.

**Requirements:**
1. Implement a skip list supporting `insert(key, value)`, `search(key)`, `update(key, value)` (insert of an existing key updates in place), `delete(key)`, and `range_scan(lo, hi)`.
2. Level promotion is probabilistic: each new node climbs a level with p = 0.5, capped at MAX_LEVEL.
3. Verify with tests: strict key ordering after 200 random inserts, search hits/misses, update-in-place, delete (including absent-key no-op), and a range scan that returns exactly the keys in `[lo, hi]`.

**Interview traps:** where the towers get re-linked on delete (every express lane that points to the victim), why expected O(log n) holds (each level halves the search space, like binary search), and why skip lists beat red-black trees for lock-friendly concurrent writes (Java's `ConcurrentSkipListMap`).

In [0]:
import random
random.seed(28)

MAX_LEVEL = 16
P = 0.5

class _Node:
    __slots__ = ("key", "val", "next")
    def __init__(self, key, val, level):
        self.key, self.val = key, val
        self.next = [None] * level           # one forward pointer per level
    @property
    def level(self):
        return len(self.next)

class SkipList:
    def __init__(self):
        self.head = _Node(None, None, MAX_LEVEL)
        self.level = 1                        # highest level currently in use

    @staticmethod
    def _random_level():
        lv = 1
        while random.random() < P and lv < MAX_LEVEL:
            lv += 1
        return lv

    def _find_predecessors(self, key):
        """update[i] = rightmost node at level i whose key < key (the splice points)."""
        update = [None] * MAX_LEVEL
        cur = self.head
        for i in range(self.level - 1, -1, -1):        # drop from the express lanes down
            while cur.next[i] is not None and cur.next[i].key < key:
                cur = cur.next[i]
            update[i] = cur
        return update

    def insert(self, key, val):
        update = self._find_predecessors(key)
        cand = update[0].next[0]
        if cand is not None and cand.key == key:       # existing key -> update in place
            cand.val = val
            return False
        lv = self._random_level()
        if lv > self.level:
            for i in range(self.level, lv):
                update[i] = self.head
            self.level = lv
        node = _Node(key, val, lv)
        for i in range(lv):                            # splice into every level of its tower
            node.next[i] = update[i].next[i]
            update[i].next[i] = node
        return True

    def search(self, key):
        cand = self._find_predecessors(key)[0].next[0]
        return cand.val if (cand is not None and cand.key == key) else None

    def delete(self, key):
        update = self._find_predecessors(key)
        cand = update[0].next[0]
        if cand is None or cand.key != key:
            return False
        for i in range(cand.level):                    # unlink from every express lane
            if update[i].next[i] is cand:
                update[i].next[i] = cand.next[i]
        while self.level > 1 and self.head.next[self.level - 1] is None:
            self.level -= 1                            # trim empty top lanes
        return True

    def range_scan(self, lo, hi):
        out = []
        cur = self._find_predecessors(lo)[0].next[0]
        while cur is not None and cur.key <= hi:
            if cur.key >= lo:
                out.append((cur.key, cur.val))
            cur = cur.next[0]
        return out

    def keys(self):
        out, cur = [], self.head.next[0]
        while cur is not None:
            out.append(cur.key)
            cur = cur.next[0]
        return out

# ---- BUILD & VERIFY ----
sl = SkipList()
keys = random.sample(range(1000), 200)
for k in keys:
    sl.insert(k, f"v{k}")
assert sl.keys() == sorted(keys)                       # strict order after 200 random inserts

for k in keys[:50]:                                    # every inserted key is findable
    assert sl.search(k) == f"v{k}"
misses = [k for k in range(1000) if k not in set(keys)]
for k in misses[:50]:                                 # absent keys return None
    assert sl.search(k) is None

k_present = keys[0]                                   # guaranteed-present key
sl.insert(k_present, "UPDATED")
assert sl.search(k_present) == "UPDATED"
assert len(sl.keys()) == 200                          # update did NOT duplicate

for k in keys[:40]:                                    # deletes remove exactly what they claim
    assert sl.delete(k) is True
    assert sl.search(k) is None
assert sl.delete(misses[0]) is False                    # absent-key delete is a no-op
assert sl.keys() == sorted(set(keys) - set(keys[:40]))

lo, hi = 250, 350
scanned = sl.range_scan(lo, hi)
expected = sorted(k for k in set(keys) - set(keys[:40]) if lo <= k <= hi)
assert [k for k, _ in scanned] == expected             # range scan = EXACTLY [lo, hi]
print("Q4 OK — skip list:", len(sl.keys()), "live keys; ordering/search/update/delete/range_scan verified")
print("Talk track: expected O(log n) — each level halves the search space; MAX_LEVEL caps the worst case;")
print("no rebalancing = fewer moving parts than RB-trees — why Redis sorted sets and Java's")
print("ConcurrentSkipListMap are built on skip lists.")

## Q5 · PySpark — Incremental AI-Function Enrichment [FAANG]

**Scenario (2025-2026 GenAI pipeline trend):** A support-ticket lakehouse runs every 15 minutes. Raw tickets land in bronze; a **SQL AI function** pass (`ai_classify` + `ai_analyze_sentiment`) enriches them into a serving table. Each AI call costs tokens, so the job must be **incremental and idempotent**: only newly-arrived rows get enriched, re-runs enrich nothing, and duplicates are impossible.

**Requirements:**
1. Bronze: `ticket_id, body` (raw). Enriched output: adds `model_version, category, sentiment`.
2. One generic incremental path: `bronze LEFT ANTI JOIN enriched` finds pending rows → AI functions enrich them → append.
3. Prove it: run 1 enriches all 5 tickets; an immediate re-run enriches **0**; after 2 new tickets arrive, exactly 2 more are enriched; 7 total rows, zero duplicate `ticket_id`s.

**Interview traps:** the lazy-read trap (counting pending rows *after* writing to the same table the anti-join reads re-evaluates against the post-write state and returns 0 — materialize the count first); and why `model_version` is stamped on every row (drift forensics when you re-train or switch prompts).

In [0]:
# bronze: raw tickets (no AI columns yet)
bronze_rows = [
    (1, "My package has not arrived and the tracking has not updated in 5 days"),
    (2, "You charged me twice for the same order on my credit card"),
    (3, "The screen arrived cracked and will not turn on"),
    (4, "How do I change my delivery address for order 5521?"),
    (5, "The app crashes every time I try to check out"),
]
spark.createDataFrame(bronze_rows, "ticket_id INT, body STRING") \
     .write.format("delta").mode("overwrite").save(f"{BASE}/q5/bronze")
# empty enriched table with the full target schema
spark.createDataFrame([], "ticket_id INT, body STRING, model_version STRING, category STRING, sentiment STRING") \
     .write.format("delta").mode("overwrite").save(f"{BASE}/q5/enriched")

def enrich_incremental():
    """Enrich ONLY rows not yet in the output (idempotent + cost guard)."""
    bronze = spark.read.format("delta").load(f"{BASE}/q5/bronze")
    done = spark.read.format("delta").load(f"{BASE}/q5/enriched").select("ticket_id")
    pending = bronze.join(done, "ticket_id", "left_anti")
    n = pending.count()               # materialize BEFORE the append (lazy-read trap)
    if n > 0:
        pending.select(
            "ticket_id", "body", F.lit("v1").alias("model_version"),
            F.expr("ai_classify(body, ARRAY('shipping', 'billing', 'product quality', 'app issue'))").alias("category"),
            F.expr("ai_analyze_sentiment(body)").alias("sentiment")) \
            .write.format("delta").mode("append").save(f"{BASE}/q5/enriched")
    return n

n1 = enrich_incremental()
assert n1 == 5, n1
n1b = enrich_incremental()             # re-run with no new data: nothing re-enriched
assert n1b == 0, n1b
assert spark.read.format("delta").load(f"{BASE}/q5/enriched").count() == 5

# 2 new tickets arrive -> exactly 2 more enriched
spark.createDataFrame([(6, "Refund still missing from my statement"),
                       (7, "Door handle snapped off the oven door")], "ticket_id INT, body STRING") \
     .write.format("delta").mode("append").save(f"{BASE}/q5/bronze")
n2 = enrich_incremental()
assert n2 == 2, n2
e = spark.read.format("delta").load(f"{BASE}/q5/enriched")
assert e.count() == 7
assert e.groupBy("ticket_id").count().filter("count > 1").count() == 0   # zero dupes -> idempotent
e.orderBy("ticket_id").show(truncate=False)
print("Q5 OK — run1=5, re-run=0, run2=2; 7 rows, zero dupes; tokens spent only on new rows")
print("Talk track: model_version per row gives drift forensics; left_anti on a stable business key;")
print("in production gate AI calls with a budget column (e.g. cap tokens per run) and land a DLQ for failures.")

## Q6 · PySpark — Higher-Order Functions on Nested Data [HARD]

**Scenario (Netflix/Adobe event-schemas style):** IoT and product analytics tables store a **month of per-customer metrics inside one row** as `ARRAY<STRUCT<ym, spend, plan>>`. Analytics needs per-element computations without exploding the array and re-aggregating — Spark's **higher-order functions** (`transform`, `aggregate`, `filter`, `exists`) run these inside the JVM, row-local, with zero shuffle.

**Requirements (all with column-expression HOFs, no UDFs, no explode):**
1. `transform`: flag each month above/below the customer's own average spend (nested `aggregate` inside `transform`).
2. `aggregate`: total spend and max monthly spend in one pass over the array.
3. `exists` / `filter`: does the customer ever hold an `enterprise` plan? how many `pro` months?
4. `transform` with `(element, index)`: element-wise month-over-month spend ratios.

**Interview trap:** when an explode + groupBy would actually be *cheaper* (data is re-aggregated across rows later anyway) vs when row-local HOFs win (per-customer windows, no shuffle, preserved nesting).

In [0]:
data = [
    ("c1", [("2026-01", 100.0, "basic"), ("2026-02", 120.0, "basic"), ("2026-03", 90.0, "pro")]),
    ("c2", [("2026-01", 10.0, "basic"), ("2026-02", 12.0, "basic"), ("2026-03", 11.0, "basic")]),
    ("c3", [("2026-01", 500.0, "pro"), ("2026-02", 480.0, "pro"), ("2026-03", 700.0, "enterprise")]),
]
spark.createDataFrame(data, "cust STRING, months ARRAY<STRUCT<ym:STRING, spend:DOUBLE, plan:STRING>>") \
     .createOrReplaceTempView("q6_cust")

res = spark.sql("""
SELECT
  cust,
  -- transform + nested aggregate: flag each month vs the customer's own average
  transform(months, m -> struct(m.ym AS ym, m.spend AS spend,
         m.spend > aggregate(months, 0D, (acc, x) -> acc + x.spend) / size(months) AS above_avg)) AS months_flagged,
  -- aggregate HOF: total & max in one pass
  aggregate(months, 0D, (acc, x) -> acc + x.spend)                        AS total_spend,
  aggregate(months, CAST(0 AS DOUBLE), (acc, x) -> greatest(acc, x.spend)) AS max_month_spend,
  -- exists / filter
  exists(months, m -> m.plan = 'enterprise')     AS has_enterprise,
  size(filter(months, m -> m.plan = 'pro'))      AS n_pro_months,
  -- transform with (element, index): month-over-month ratios
  transform(months, (m, i) -> CASE WHEN i = 0 THEN NULL
                                   ELSE m.spend / months[i - 1].spend END) AS mom_ratios
FROM q6_cust
ORDER BY cust
""")
res.show(truncate=False)
got = {r["cust"]: r for r in res.collect()}

# VERIFY — every HOF result equals a python reference over the same data
for c, months in data:
    tot = sum(m[1] for m in months)
    assert abs(got[c]["total_spend"] - tot) < 1e-9
    assert abs(got[c]["max_month_spend"] - max(m[1] for m in months)) < 1e-9
    ratios = [None] + [months[i][1] / months[i-1][1] for i in range(1, len(months))]
    for a, b in zip(got[c]["mom_ratios"], ratios):
        assert (a is None and b is None) or abs(a - b) < 1e-9
    for flag, m in zip(got[c]["months_flagged"], months):
        assert flag["above_avg"] == (m[1] > tot / len(months))
assert got["c1"]["n_pro_months"] == 1 and got["c2"]["n_pro_months"] == 0
assert got["c3"]["has_enterprise"] is True and got["c2"]["has_enterprise"] is False
print("Q6 OK — transform/aggregate/filter/exists/index-transform all match python reference")
print("Talk track: HOFs are whole-stage codegen'd and row-local (no shuffle); UDFs cross the JVM-Python")
print("boundary per row; explode wins only when you must re-aggregate across customers afterward.")

## Q7 · Spark Optimization — Dynamic Resource Allocation Policy [HARD]

**Scenario (Meta/Netflix infra style):** Your jobs run on shared clusters where idle executors burn money. You are asked to design (and defend) a **dynamic resource allocation (DRA)** policy: when do executors get added, when removed, and what happens to in-flight shuffle data when they go.

**Requirements:**
1. Explain the scale-up trigger (pending/backlogged tasks), scale-down trigger (`executorIdleTimeout`), and the safety floors/ceilings (`minExecutors`, `maxExecutors`, `spark.dynamicAllocation.schedulerBacklogTimeout`).
2. **Simulate** the policy: given a burst-then-quiet workload, show the executor pool growing to cap and shrinking back to min after the idle timeout — and show a too-short idle timeout thrashing the pool.
3. Explain the shuffle-data problem on executor removal (`shuffleTracking.enabled`, external shuffle service, cache decommissioning) and the recommended configs.

**Interview trap:** what DRA does to cached blocks and shuffle files when it removes an executor — and why `shuffleTracking.enabled` + external shuffle service is the modern answer on Databricks.

In [0]:
def simulate_dra(arrival_rate, proc_rate_per_exec, idle_timeout_s, horizon_s,
                 min_exec=1, max_exec=10, start_exec=2, tick=10):
    """Simulate an executor pool under DRA: scale up on backlog, down after idle timeout."""
    execs, backlog, history, last_busy = start_exec, 0.0, [], None
    for t in range(0, horizon_s, tick):
        backlog += arrival_rate(t) * tick
        capacity = execs * proc_rate_per_exec * tick
        done = min(backlog, capacity)
        backlog -= done
        if done > 0:
            last_busy = t
        if backlog > 0 and execs < max_exec:      # scale-up: backlog exists
            execs += 1
        elif last_busy is not None and t - last_busy >= idle_timeout_s and execs > min_exec:
            execs -= 1                             # idle long enough -> release
        history.append((t, execs, round(backlog, 2)))
    return history

# burst then quiet: pool grows to cap during the burst, drains, shrinks after the idle timeout
h = simulate_dra(arrival_rate=lambda t: 5.0 if t < 100 else 0.0, proc_rate_per_exec=0.5,
                 idle_timeout_s=60, horizon_s=400)
peak_execs, final_execs, final_backlog = max(x[1] for x in h), h[-1][1], h[-1][2]
assert peak_execs == 10 and final_execs == 1 and final_backlog == 0
print(f"Q7a OK — DRA sim: peak {peak_execs} execs during burst -> settles at {final_execs} after idle timeout, backlog drained")

# too-short idle timeout: the pool thrashes (repeated release/re-request risks shuffle loss)
h2 = simulate_dra(arrival_rate=lambda t: 5.0 if t < 100 else 0.0, proc_rate_per_exec=0.5,
                  idle_timeout_s=10, horizon_s=400)
releases = sum(1 for a, b in zip(h2, h2[1:]) if b[1] < a[1])
assert releases > 0
print(f"Q7b OK — short idle timeout releases {releases} times (thrash; shuffle files can be lost per release)")

# conf surface (values are cluster-policy managed on serverless — show how to read them safely)
def safe_conf(key):
    try:
        return spark.conf.get(key)
    except Exception:
        return "CONFIG_NOT_AVAILABLE (managed on this compute)"
for k in ["spark.dynamicAllocation.enabled", "spark.dynamicAllocation.executorIdleTimeout",
          "spark.dynamicAllocation.minExecutors", "spark.dynamicAllocation.maxExecutors",
          "spark.dynamicAllocation.shuffleTracking.enabled"]:
    print(f"  {k} = {safe_conf(k)}")
print("Q7 OK — DRA decision math + conf surface verified")
print("Talk track: removal must be shuffle-safe — keep shuffleTracking.enabled + external shuffle service;")
print("cache decommissioning writes RDD blocks out before exit; minExecutors>0 protects against cold starts;")
print("on autoscaling clusters prefer queue-aware policy (backlogTimeout) over naive task-pending counts.")

## Q8 · Spark Optimization — Lineage, Caching & Checkpoint Truncation [HARD]

**Scenario (long-running pipeline review):** A nightly job builds a DataFrame through 40 chained transformations, reuses it 3 times, and runs slower every week. The review asks: what's wrong with the plan, what does `cache()`/`persist()` actually fix, what does `checkpoint()` fix that caching doesn't, and what do the configs mean?

**Requirements:**
1. **Demonstrate** lineage growth: build ever-deeper DataFrame chains and show the physical plan growing with depth.
2. Explain the three truncation tools: `cache()` (recompute avoidance, in-memory, lineage intact), `localCheckpoint()` (truncate lineage, storage survives executor loss — unreliable), `checkpoint()` (truncate lineage **and** survive failure, writes to reliable storage).
3. Show where the relevant configs live (`spark.sql.checkpointInterval`, `spark.task.maxFailures`) and how to read them safely on managed compute.

**Interview trap:** caching does NOT truncate lineage — a 40-deep plan still serializes, ships to executors, and plans on the driver every run; checkpoints are what actually cut the DAG. Also: `checkpoint()` on Spark Connect/serverless has JVM limitations — know your fallbacks.

In [0]:
import io, contextlib

def plan_len(d):
    """Length of the physical plan text (proxy for plan size the driver must build every run)."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        d.explain("simple")
    return len(buf.getvalue())

# grow a lineage 1 -> 40 transformations deep and watch the plan grow
results, cur = [], spark.range(1000).withColumn("x", F.col("id") * 2)
for i in range(1, 41):
    cur = cur.withColumn(f"c{i}", F.col("x") + i)
    if i in (1, 5, 10, 20, 40):
        results.append((i, plan_len(cur)))
print("depth -> plan chars:", results)
chars = [r[1] for r in results]
assert all(chars[i] < chars[i+1] for i in range(len(chars)-1))   # monotone growth
assert chars[-1] / chars[0] > 1.5                                # material growth
print(f"Q8a OK — physical plan grows {chars[-1]/chars[0]:.1f}x from depth 1 -> 40;")
print("      driver pays plan-build cost every run, and a lost executor replays the WHOLE lineage")

# where the truncation configs live (safe reads on managed compute)
def safe_conf(key):
    try:
        return spark.conf.get(key)
    except Exception:
        return "CONFIG_NOT_AVAILABLE (managed on this compute)"
for k in ["spark.sql.checkpointInterval", "spark.task.maxFailures", "spark.sql.shuffle.partitions"]:
    print(f"  {k} = {safe_conf(k)}")
print("Q8 OK — lineage growth demonstrated; conf surface readable")
print("Talk track: cache() avoids recompute but keeps lineage (plan still ships, still replays on failure);")
print("localCheckpoint() truncates lineage but dies with the executor; checkpoint() truncates AND survives")
print("failure by writing to reliable storage — the only true DAG cut. On Spark Connect/serverless,")
print("DataFrame.checkpoint can hit JVM limits — the fallback is write-to-Delta-then-read-back, which")
print("gives the same truncation (a table scan has no history) plus transactional durability.")
print("Recompute cascade: any shuffle stage failure reruns every upstream stage since the last shuffle.")

## Q9 · ETL — Debezium-Style Log-Based CDC [FAANG]

**Scenario (Netflix/Uber migration interview classic):** You replace nightly full loads from MySQL with **log-based CDC via Debezium + Kafka**. No dual-writes, no source queries — the binlog is the source of truth. Walk the whole mechanism end to end.

**Requirements:**
1. Explain the **consistent snapshot** phase: how Debezium captures a binlog position, takes a schema+data snapshot, then tails from the captured offset without missing or double-applying changes.
2. Replay a simulated binlog (`c`/`u`/`d` ops with LSN ordering, partial after-images) onto a snapshot state and prove it converges to the source's current state.
3. Prove **idempotency** (duplicate tail delivery changes nothing — last-write-wins by LSN) and explain **heartbeats** (why quiet tables still need offset advancement).
4. Map the Kafka topic layout (`<server>.<db>.<table>`), partitioning by primary key, and how ordering is guaranteed per key.

**Interview traps:** why updates must merge with existing state (partial after-images), why offset capture must happen BEFORE the snapshot lock releases, and what breaks when a quiet table's offset falls behind binlog retention.

In [0]:
# simulated ordered binlog (what Debezium emits, minus the envelope):
# (lsn, op, pk, after) — op: c=create, u=update, d=delete; updates carry ONLY changed columns
binlog = [
    (101, "c", 1, {"cust_id": 1, "email": "a@x.com", "tier": "basic"}),
    (102, "c", 2, {"cust_id": 2, "email": "b@x.com", "tier": "basic"}),
    (103, "u", 1, {"cust_id": 1, "tier": "pro"}),
    (104, "c", 3, {"cust_id": 3, "email": "c@x.com", "tier": "basic"}),
    (105, "d", 2, None),
    (106, "u", 3, {"cust_id": 3, "tier": "enterprise"}),
]

# phase 1 — consistent snapshot: read current source state + capture the binlog position
# while the snapshot lock is held (BEFORE releasing it, so no change is missed)
source_state_at_snapshot = {1: {"cust_id": 1, "email": "a@x.com", "tier": "pro"},
                            3: {"cust_id": 3, "email": "c@x.com", "tier": "basic"}}
snapshot_lsn = 103

# phase 2 — incremental tail from the captured position
tail = [(lsn, op, pk, after) for lsn, op, pk, after in binlog if lsn > snapshot_lsn]

def apply_log(state, events):
    """Apply c/u/d with last-write-wins by LSN; updates MERGE partial after-images."""
    for lsn, op, pk, after in sorted(events, key=lambda e: e[0]):   # per-key ordering
        if op == "d":
            state.pop(pk, None)
        else:
            row = state.setdefault(pk, {"cust_id": pk})
            if after:
                row.update(after)   # merge, because updates carry only changed columns
    return state

state = {k: dict(v) for k, v in source_state_at_snapshot.items()}
apply_log(state, tail)
assert state == {1: {"cust_id": 1, "email": "a@x.com", "tier": "pro"},
                 3: {"cust_id": 3, "email": "c@x.com", "tier": "enterprise"}}, state
print("Q9a OK — snapshot + tail replays to the source's true state:", state)

# at-least-once delivery: the SAME tail delivered twice changes nothing
state2 = {k: dict(v) for k, v in source_state_at_snapshot.items()}
apply_log(state2, tail + tail)
assert state2 == state
print("Q9b OK — duplicate tail delivery is idempotent (LSN last-write-wins)")

# heartbeat events keep the connector's offset advancing on idle tables
heartbeats = [(107, "h", None, None)]
max_seen = max(e[0] for e in tail + heartbeats)
assert max_seen == 107
print("Q9c OK — heartbeat advances the offset so a quiet table's position doesn't fall")
print("      behind binlog retention (else: offset out of range -> full re-snapshot)")

# materialize the converged state with Spark (the sink side of the pipeline)
tgt = spark.createDataFrame([(v["cust_id"], v["email"], v["tier"]) for v in state.values()],
                            "cust_id INT, email STRING, tier STRING")
assert tgt.count() == 2
tgt.show()
print("Q9 OK — Debezium-style CDC: snapshot + tail + heartbeat + idempotent apply verified")
print("Talk track: topics are <server>.<db>.<table>, partitioned by PRIMARY KEY (per-key ordering);")
print("snapshot lock = MySQL FTWRL / Postgres replication slot; envelopes carry before/after/source/lsn;")
print("sink = MERGE on pk or append-only event log (replayable), never naive upserts without keys.")

## Q10 · ETL — Deterministic HMAC PII Tokenization [HARD]

**Scenario (fintech/health data platform):** Regulation forbids raw emails in analytics tables, but the `customers` and `orders` tables share the email join key — so the tokenization must be **deterministic** (same input → same token) to preserve referential integrity, **key-dependent** (not a plain hash — rainbow tables can't reverse it), and **rotatable** (re-tokenize under a new key version without exposing raw PII downstream).

**Requirements:**
1. Tokenize emails on both sides of the join with a versioned HMAC-SHA256 (salted master key from a secret scope; truncated token format `tok_<key_version>_<digest>`).
2. Prove the tokenized join still answers the business question (per-customer revenue).
3. Prove the security properties: deterministic, distinct inputs → distinct tokens, not equal to a plain SHA-256 of the input.
4. **Key rotation**: via a vault table (the only place with raw PII mapping v1→v2 tokens), re-tokenize downstream without leaking raw values.

**Interview traps:** why deterministic tokenization defeats the naive "just hash it" (unsalted SHA-256 is trivially rainbow-tabled); where the vault table lives and who can read it; salt vs key versioning.

In [0]:
import hmac, hashlib
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

MASTER_KEY = b"d28-demo-master-key"          # production: dbutils.secrets.get(scope, key)
KEYS = {1: MASTER_KEY + b"|v1", 2: MASTER_KEY + b"|v2"}   # versioned keys

def hmac_token(key_version, value):
    key = KEYS[key_version]
    mac = hmac.new(key, value.encode(), hashlib.sha256).hexdigest()[:16]
    return f"tok_{key_version}_{mac}"

tokenize  = udf(lambda e: hmac_token(1, e), StringType())
tokenize2 = udf(lambda e: hmac_token(2, e), StringType())

# two source tables sharing the email join key
customers = [("alice@example.com", "Alice"), ("bob@example.com", "Bob"), ("carol@example.com", "Carol")]
orders     = [("alice@example.com", 250.0), ("bob@example.com", 100.0), ("alice@example.com", 80.0)]
c = spark.createDataFrame(customers, "email STRING, name STRING")
o = spark.createDataFrame([(e, float(a)) for e, a in orders], "email STRING, amount DOUBLE")

# tokenize BOTH sides with the same key version -> join survives, raw PII does not
ct = c.withColumn("email_token", tokenize("email")).drop("email")
ot = o.withColumn("email_token", tokenize("email")).drop("email")
joined = ct.join(ot, "email_token")
assert joined.count() == 3
spent = {r["name"]: r["total"] for r in joined.groupBy("name").agg(F.sum("amount").alias("total")).collect()}
assert abs(spent["Alice"] - 330.0) < 1e-9 and abs(spent["Bob"] - 100.0) < 1e-9
print("Q10a OK — tokenized join preserves analytics: Alice", spent["Alice"], "/ Bob", spent["Bob"])

# security properties
assert hmac_token(1, "alice@example.com") == hmac_token(1, "alice@example.com")   # deterministic
assert hmac_token(1, "alice@example.com") != hmac_token(1, "bob@example.com")     # distinct
assert hmac_token(1, "alice@example.com") != \
       "tok_1_" + hashlib.sha256(b"alice@example.com").hexdigest()[:16]            # not a plain hash
print("Q10b OK — deterministic, key-dependent, rainbow-table-resistant")

# key rotation via the vault (the ONLY table allowed to hold raw PII / both versions)
vault = c.withColumn("t1", tokenize("email")).withColumn("t2", tokenize2("email")).drop("email")
vault.show(truncate=False)
rotated = ot.join(vault.select("t1", "t2"), ot.email_token == vault.t1, "left") \
            .select("t2", "amount")
assert rotated.filter("t2 IS NULL").count() == 0     # every v1 token found a mapping
assert rotated.count() == 3
assert all(r["t1"] != r["t2"] for r in vault.collect())  # v1 != v2 for the same identity
# downstream carries only tokens — never raw PII
assert set(ct.columns) == {"email_token", "name"} and set(ot.columns) == {"email_token", "amount"}
print("Q10c OK — key rotation re-tokenizes via vault join without exposing raw PII downstream")
print("Q10 OK — HMAC tokenization pipeline verified")
print("Talk track: unsalted hashes fall to rainbow tables (LinkedIn 2012); HMAC adds a SECRET key;")
print("store the vault in a locked catalog/schema with row-level security; key rotation = re-tokenize via")
print("vault join, never a full PII round-trip through the analytics estate.")

## Q11 · ETL — Metadata-Driven Ingestion Framework [HARD]

**Scenario (500-source enterprise):** Hand-written per-source pipelines cannot scale to 500+ feeds. Design a **metadata-driven ingestion framework**: every pipeline is a *row in a registry*, and one generic code path executes all of them — onboarding a new source becomes an INSERT, not a code deployment.

**Requirements:**
1. A source registry: name, ingest archetype (`append_s3` with watermark vs `full_file` overwrite), target, watermark column, schedule, SLA, owner.
2. ONE generic `run_ingest(source)` that reads registry metadata: watermark-filtered incremental append OR idempotent full-file overwrite — no per-source code.
3. A control table for watermark state; prove that advancing the watermark makes re-runs ingest 0 rows.
4. Registry self-validation: SLA bounds and schedule-shape checks gate config changes.

**Interview traps:** where the watermark lives (control table, not re-computed from the target); why `full_file` sources must overwrite, never append; what breaks when two archetypes share one code path; schema evolution and how the registry records it.

In [0]:
registry = [
    # source, ingest_type, watermark_col, schedule, sla_min, owner
    ("crm_users",   "append_s3", "updated_at", "*/15 * * * *",  30, "growth-eng"),
    ("shop_orders", "append_s3", "created_at", "*/5 * * * *",  10, "commerce"),
    ("ledger",      "full_file", None,          "0 * * * *",    60, "finops"),
]
reg = spark.createDataFrame(registry, "source STRING, ingest_type STRING, watermark_col STRING, schedule STRING, sla_min INT, owner STRING")
reg.show(truncate=False)

# simulated raw drops for the two incremental sources (parquet, as S3/ADLS would deliver)
spark.createDataFrame([("u1", "2026-11-19T09:00:00"), ("u2", "2026-11-19T09:10:00")], "uid STRING, updated_at STRING") \
     .withColumn("updated_at", F.to_timestamp("updated_at")) \
     .write.mode("overwrite").parquet(f"{BASE}/q11/raw/crm_users")
spark.createDataFrame([(1, "2026-11-19T09:02:00", 25.0), (2, "2026-11-19T09:03:00", 40.0)], "oid INT, created_at STRING, amt DOUBLE") \
     .withColumn("created_at", F.to_timestamp("created_at")) \
     .write.mode("overwrite").parquet(f"{BASE}/q11/raw/shop_orders")

# control table: per-source watermark state (persisted between runs in production)
state = {"crm_users": "2026-11-19T08:00:00", "shop_orders": "2026-11-19T09:00:00", "ledger": None}

def run_ingest(source):
    """ONE generic code path driven entirely by registry metadata."""
    cfg = next(r for r in registry if r[0] == source)
    _, ingest_type, wm_col = cfg[0], cfg[1], cfg[2]
    raw = spark.read.parquet(f"{BASE}/q11/raw/{source}")
    if ingest_type == "append_s3":
        low = state[source]
        raw = raw.filter(F.col(wm_col) > F.to_timestamp(F.lit(low)))   # incremental cut
        n = raw.count()
        if n > 0:
            new_wm = raw.agg(F.max(wm_col)).collect()[0][0]            # advance watermark
            raw.write.format("delta").mode("append").save(f"{BASE}/q11/silver/{source}")
            state[source] = new_wm.strftime("%Y-%m-%dT%H:%M:%S")
        return n
    elif ingest_type == "full_file":
        n = raw.count()
        raw.write.format("delta").mode("overwrite").save(f"{BASE}/q11/silver/{source}")  # naturally idempotent
        return n

n_crm, n_shop = run_ingest("crm_users"), run_ingest("shop_orders")
assert n_crm == 2 and n_shop == 2, (n_crm, n_shop)
n_crm_again = run_ingest("crm_users")             # watermark advanced -> 0 new rows
assert n_crm_again == 0
print("Q11a OK — registry-driven ingest: 2+2 rows, watermark advances, re-run ingests 0 (idempotent)")

# full_file archetype: overwrite re-load is naturally idempotent
spark.createDataFrame([(1, 300.0), (2, 500.0)], "acct INT, bal DOUBLE") \
     .write.mode("overwrite").parquet(f"{BASE}/q11/raw/ledger")
assert run_ingest("ledger") == 2
assert run_ingest("ledger") == 2
l = spark.read.format("delta").load(f"{BASE}/q11/silver/ledger")
assert l.count() == 2 and l.groupBy("acct").count().filter("count > 1").count() == 0
print("Q11b OK — full-file source: overwrite re-load idempotent, no dupes")

# registry self-validation gates bad config before any pipeline reads it
checks = reg.withColumn("sla_ok", F.col("sla_min").between(5, 1440)) \
            .withColumn("sched_ok", F.length("schedule") >= 6)
assert checks.filter("NOT sla_ok OR NOT sched_ok").count() == 0
print("Q11c OK — registry self-validation (SLA bounds + schedule shape) gates config changes")
print("Q11 OK — metadata-driven framework: 3 sources, 2 archetypes, one code path")
print("Talk track: onboarding = INSERT a row; schema evolution = registry column metadata +;")
print("per-archetype validators (NOT per-source code); control table is the source of truth for watermarks,")
print("never re-derive from the target (late data arrives below max); full_file must overwrite or you double-load.")

## Q12 · Data Modeling — Ratio & Non-Additive Metrics [HARD]

**Scenario (metric-layer design review):** Your dimensional model feeds a BI layer. An analyst reports the dashboard's **conversion rate** disagrees with finance's number. Root cause: someone stored a pre-computed ratio at a lower grain, then **averaged the ratios** at the dashboard grain — a non-additive metric aggregated incorrectly.

**Requirements:**
1. Show the trap live: `AVG(orders/sessions)` per region vs `SUM(orders)/SUM(sessions)` per day — prove they differ, on every day of the sample.
2. Same rule for money: weighted **AOV** — naive average of regional AOVs vs revenue/orders.
3. Share-of-total metrics: show shares computed at two different grains (region, day), that each sums to 1 **within its own slice** but must never be added across slices.
4. State the design rule: facts store **numerators and denominators**, never ratios; ratios are derived at query time at the display grain.

**Interview trap:** when a stored ratio IS acceptable (a true snapshot ratio at its own grain, never re-aggregated), and the semi-additive trap (a ratio-of-sums across a different dimension than intended).

In [0]:
# part 1 — conversion rate: numerator/denominator facts at region-day grain
facts = [
    ("2026-11-18", "us", 100, 200),
    ("2026-11-18", "eu",  20,  50),
    ("2026-11-19", "us",  90, 150),
    ("2026-11-19", "eu",  40,  80),
]
spark.createDataFrame([(d, r, int(o), int(s)) for d, r, o, s in facts],
                      "dt STRING, region STRING, orders INT, sessions INT") \
     .createOrReplaceTempView("q12_facts")

res = spark.sql("""
WITH per_day AS (
  SELECT dt, SUM(orders) AS orders, SUM(sessions) AS sessions,
         AVG(orders / sessions) AS avg_of_ratios        -- the trap: averaging regional rates
  FROM q12_facts GROUP BY dt
)
SELECT dt, orders, sessions, avg_of_ratios,
       CAST(orders AS DOUBLE) / sessions AS ratio_of_sums  -- correct: re-derive at display grain
FROM per_day ORDER BY dt
""")
res.show()
nov18, nov19 = res.collect()
assert abs(nov18["avg_of_ratios"] - 0.45) < 1e-9 and abs(nov18["ratio_of_sums"] - 0.48) < 1e-9
assert abs(nov19["avg_of_ratios"] - 0.55) < 1e-9 and abs(nov19["ratio_of_sums"] - 130.0/230.0) < 1e-9
for r in (nov18, nov19):
    assert abs(r["avg_of_ratios"] - r["ratio_of_sums"]) > 1e-6     # trap is real on BOTH days
print("Q12a OK — conversion rate: avg-of-ratios 0.45 vs ratio-of-sums 0.48 (and daily 0.55 vs 0.565)")

# part 2 — weighted AOV: naive average of regional AOVs is wrong when volumes are unbalanced
f2 = [
    ("2026-11-18", "us", 1, 500.0), ("2026-11-18", "us", 2, 100.0),
    ("2026-11-18", "eu", 3, 900.0),
    ("2026-11-19", "us", 4, 50.0), ("2026-11-19", "eu", 5, 40.0),
]
spark.createDataFrame([(d, r, int(o), float(a)) for d, r, o, a in f2],
                      "dt STRING, region STRING, oid INT, amt DOUBLE") \
     .createOrReplaceTempView("q12_orders")
res2 = spark.sql("""
WITH regional AS (
  SELECT dt, region, COUNT(*) AS n, SUM(amt) AS rev
  FROM q12_orders GROUP BY dt, region
)
SELECT dt, SUM(n) AS n_orders, SUM(rev) AS revenue,
       CAST(SUM(rev) AS DOUBLE) / SUM(n)   AS correct_aov,
       AVG(CAST(rev AS DOUBLE) / n)        AS naive_avg_of_regional_aovs
FROM regional GROUP BY dt ORDER BY dt
""")
res2.show()
r18, r19 = res2.collect()
assert abs(r18["correct_aov"] - 500.0) < 1e-9 and abs(r18["naive_avg_of_regional_aovs"] - 600.0) < 1e-9
assert abs(r19["correct_aov"] - 45.0) < 1e-9 and abs(r19["naive_avg_of_regional_aovs"] - 45.0) < 1e-9
leaf = spark.sql("SELECT AVG(amt) AS leaf_aov, SUM(amt)/COUNT(*) AS fact_aov FROM q12_orders").collect()[0]
assert abs(leaf["leaf_aov"] - leaf["fact_aov"]) < 1e-9     # equal only at the LEAF grain
print("Q12b OK — AOV: ratio-of-sums 500 vs avg-of-regional-AOVs 600 on Nov 18; equal on balanced Nov 19")

# part 3 — share-of-total: sums to 1 within its own slice, meaningless across slices
res3 = spark.sql("""
WITH total AS (SELECT SUM(orders) AS all_orders FROM q12_facts),
     by_region AS (SELECT region AS slice_val, SUM(orders) AS orders FROM q12_facts GROUP BY region),
     by_day AS (SELECT dt AS slice_val, SUM(orders) AS orders FROM q12_facts GROUP BY dt)
SELECT 'region' AS slice, slice_val,
       CAST(orders AS DOUBLE) / (SELECT all_orders FROM total) AS share
FROM by_region
UNION ALL
SELECT 'day', slice_val, CAST(orders AS DOUBLE) / (SELECT all_orders FROM total) FROM by_day
""")
res3.show()
sh = res3.collect()
assert abs(sum(r["share"] for r in sh if r["slice"] == "region") - 1.0) < 1e-9
assert abs(sum(r["share"] for r in sh if r["slice"] == "day") - 1.0) < 1e-9
print("Q12c OK — shares sum to 1 within each slice; adding shares ACROSS slices is meaningless")
print("Q12 OK — ratio/non-additive metric rules verified")
print("Talk track: store numerators & denominators; derive ratios at the display grain; a stored ratio")
print("is only safe if it's a true snapshot at its own grain and is never re-aggregated;")
print("semi-additive trap: ratio-of-sums across the WRONG dimension is just as wrong as avg-of-ratios.")

## Q13 · Data Modeling — SaaS Recurring-Billing & MRR Waterfall [FAANG]

**Scenario (Stripe/Chargebee-style data model):** Model a SaaS subscription business. Revenue comes from **recurring subscriptions** with expansions, contractions, and churn — the classic **MRR waterfall** (new + expansion − contraction − churned = net new MRR). This is the *canonical* non-trivial fact-modeling interview question.

**Requirements:**
1. Design the event fact: **immutable rows** for `signup` (+), `expand` (+), `contract` (−), `churn` (−), each carrying `mrr_delta` and an idempotent event id.
2. Compute the **monthly waterfall** (new/expansion/contraction/churned/net-new MRR).
3. Compute **end-of-period active MRR** — a *semi-additive* measure you must snapshot, not sum: use as-of cumulative windows to get each subscription's value at each month end.
4. Prove two identities: end-of-period active MRR == a Python replay of the same events for every month; and cumulative net-new MRR == the latest month-end active MRR.

**Interview traps:** why MRR is semi-additive (never `SUM` deltas across time in a BI tool and call it month-end MRR); storing deltas vs storing rates (deltas compose, rates don't); as-of joins to get point-in-time state; how refunds/usage-based billing complicate the model.

In [0]:
import datetime as dt
events = [
    (1, "s1", "2026-01-10", "signup",   100.0),
    (2, "s2", "2026-02-15", "signup",   250.0),
    (3, "s3", "2026-03-20", "signup",    50.0),
    (4, "s1", "2026-05-01", "expand",    40.0),   # s1: 100 -> 140
    (5, "s4", "2026-06-01", "signup",   300.0),
    (6, "s2", "2026-07-15", "contract", -50.0),   # s2: 250 -> 200
    (7, "s3", "2026-08-01", "churn",    -50.0),   # s3 -> 0
    (8, "s5", "2026-09-01", "signup",    80.0),
    (9, "s1", "2026-10-01", "churn",  -140.0),   # s1 -> 0
    (10, "s4", "2026-10-20", "expand",  120.0),   # s4: 300 -> 420
]
rows = [(i, s, dt.date.fromisoformat(d), k, float(m)) for i, s, d, k, m in events]
spark.createDataFrame(rows, "event_id INT, sub_id STRING, event_date DATE, kind STRING, mrr_delta DOUBLE") \
     .createOrReplaceTempView("q13_events")

# 1) monthly MRR movement waterfall
wf = spark.sql("""
SELECT date_trunc('month', event_date) AS month,
       SUM(CASE WHEN kind='signup'   THEN mrr_delta ELSE 0 END) AS new_mrr,
       SUM(CASE WHEN kind='expand'   THEN mrr_delta ELSE 0 END) AS expansion_mrr,
       SUM(CASE WHEN kind='contract' THEN mrr_delta ELSE 0 END) AS contraction_mrr,
       SUM(CASE WHEN kind='churn'    THEN mrr_delta ELSE 0 END) AS churned_mrr,
       SUM(mrr_delta) AS net_new_mrr
FROM q13_events GROUP BY 1 ORDER BY 1
""")
wf.show()
w = {r["month"].strftime("%Y-%m"): r for r in wf.collect()}
assert abs(w["2026-01"]["new_mrr"] - 100.0) < 1e-9
assert abs(w["2026-08"]["churned_mrr"] - (-50.0)) < 1e-9
assert abs(w["2026-10"]["net_new_mrr"] - (-20.0)) < 1e-9        # -140 churn + 120 expansion
print("Q13a OK — monthly waterfall: new/expansion/contraction/churned/net-new from deltas")

# 2) end-of-period ACTIVE MRR — semi-additive: as-of cumulative window per subscription,
#    then take each sub's LAST event up to each month end (MAX(struct) = argmax in Spark SQL)
eom = spark.sql("""
WITH active_delta AS (
  SELECT event_id, sub_id, event_date, mrr_delta,
         SUM(mrr_delta) OVER (PARTITION BY sub_id ORDER BY event_date, event_id
                              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_mrr
  FROM q13_events
),
periods AS (SELECT explode(sequence(DATE'2026-01-01', DATE'2026-11-01', INTERVAL 1 MONTH)) AS month_start),
last_evt AS (
  SELECT p.month_start, a.sub_id,
         MAX(struct(a.event_date, a.event_id)) AS last_key
  FROM periods p JOIN active_delta a
    ON a.event_date <= last_day(p.month_start)
  GROUP BY p.month_start, a.sub_id
)
SELECT l.month_start, SUM(a2.cum_mrr) AS active_mrr
FROM last_evt l JOIN active_delta a2
  ON a2.sub_id = l.sub_id AND struct(a2.event_date, a2.event_id) = l.last_key
GROUP BY l.month_start ORDER BY l.month_start
""")
eom.show()
snap = {r["month_start"].strftime("%Y-%m"): float(r["active_mrr"]) for r in eom.collect()}

# VERIFY — python replay of the same events, snapshotted at every month end
def reference_eom():
    out = {}
    for m in range(1, 12):
        month_end = dt.date(2026, m, 28 if m == 2 else 30 if m in (4, 6, 9, 11) else 31)
        subs = {}
        for i, s, d, k, delta in sorted(rows, key=lambda r: (r[2], r[0])):
            if d <= month_end:
                subs[s] = subs.get(s, 0.0) + delta
        out[f"2026-{m:02d}"] = sum(subs.values())
    return out
ref = reference_eom()
assert set(snap) == set(ref)
for m in ref:
    assert abs(snap[m] - ref[m]) < 1e-9, (m, snap[m], ref[m])
print("Q13b OK — end-of-period active MRR == python replay for ALL 11 months:", snap)

# 3) identity: cumulative net-new == latest month-end active MRR
tot = float(wf.agg(F.sum("net_new_mrr")).collect()[0][0])
assert abs(tot - snap["2026-11"]) < 1e-9 and abs(tot - snap["2026-10"]) < 1e-9
print("Q13c OK — cumulative net-new MRR == month-end active MRR:", tot)
print("Q13 OK — recurring-billing event model + MRR waterfall verified")
print("Talk track: deltas compose, rates don't (that's why we store mrr_delta, never monthly rate);")
print("MAX(struct(...)) is the idiomatic argmax; BI tools must never SUM deltas across time as MRR;")
print("usage-based/hybrid revenue needs a separate metering fact and a bridge to committed MRR.")

## Q14 · Snowflake — ACCESS_HISTORY Governance Forensics [FAANG]

**Scenario (Snowflake security audit):** A regulated customer asks: *who has read the `email` column of `customers` in the last 90 days?* And a second question: *which columns of a restricted table are actually used — so we can tighten row/column policies without breaking anyone?* Both are answered from `ACCESS_HISTORY` (a Snowflake `ACCOUNT_USAGE` view) — Snowflake's equivalent of querying access logs with **full query→column-level lineage**.

**Requirements:**
1. Write the audit query: users + timestamps that accessed a PII column, via `LATERAL FLATTEN` over `objects_accessed` → `columns`.
2. Explain the `objects_accessed` / `direct_objects_accessed` distinction (views propagate lineage to base tables — the whole point).
3. Show the columns-actually-used analysis to justify column masking (dynamic data masking) and row access policies.
4. Know the caveats: ~45-minute lag for ACCOUNT_USAGE, retention (365 days), and why `QUERY_HISTORY` alone is insufficient (it has table names, not columns).

**Interview trap:** `objects_accessed` vs `direct_objects_accessed` — the non-direct version resolves through views/secure views to the base tables, which is what governance needs; also `base_objects_accessed` in newer views.

In [0]:
# ---- Reference SQL (run on Snowflake; printed here as the interview deliverable) ----
audit_sql = """
-- Who touched the email column in the last 90 days? (PII audit)
SELECT ah.query_id, ah.user_name, ah.query_start_time,
       oa.value:objectName::string  AS table_name,
       c.value:columnName::string   AS column_name
FROM snowflake.account_usage.access_history ah,
     LATERAL flatten(input => ah.objects_accessed) oa,
     LATERAL flatten(input => oa.value:columns) c
WHERE ah.query_start_time >= dateadd(day, -90, current_timestamp())
  AND c.value:columnName::string ILIKE 'email'
ORDER BY ah.query_start_time DESC;
"""
print(audit_sql)

unused_columns_sql = """
-- Which columns of a restricted table are ACTUALLY read? (mask what isn't needed)
SELECT c.value:columnName::string AS column_name, COUNT(*) AS n_reads
FROM snowflake.account_usage.access_history ah,
     LATERAL flatten(input => ah.objects_accessed) oa,
     LATERAL flatten(input => oa.value:columns) c
WHERE ah.query_start_time >= dateadd(day, -90, current_timestamp())
  AND oa.value:objectName::string = 'MY_DB.PII.CUSTOMERS'
GROUP BY 1 ORDER BY n_reads DESC;
"""
print(unused_columns_sql)

# ---- runnable Spark analogue: column-level lineage from a query log ----
# objects_accessed resolves views to BASE tables + base columns (what governance needs);
# direct_objects_accessed lists only what the query NAMED (view alias columns).
# q2 queries the view v_customer_orders, which renames customers.email -> 'contact'.
access_log = [
    ("q1", "analyst_a",
     [{"table": "customers", "columns": ["id", "email", "tier"]},
      {"table": "orders", "columns": ["oid", "cust_id", "amount"]}],
     [{"table": "customers", "columns": ["id", "email", "tier"]},
      {"table": "orders", "columns": ["oid", "cust_id", "amount"]}]),
    ("q2", "analyst_b",
     [{"table": "customers", "columns": ["id", "email", "amount"]}],            # resolved to base
     [{"table": "v_customer_orders", "columns": ["id", "contact", "amount"]}]),   # view alias
    ("q3", "analyst_a",
     [{"table": "orders", "columns": ["oid", "amount"]}],
     [{"table": "orders", "columns": ["oid", "amount"]}]),
]
sch = "query_id STRING, user STRING, objects ARRAY<STRUCT<table:STRING, columns:ARRAY<STRING>>>, direct_objects ARRAY<STRUCT<table:STRING, columns:ARRAY<STRING>>>"
al = spark.createDataFrame(access_log, sch)
al.createOrReplaceTempView("q14_access_history")

# lineage-aware audit (objects_accessed): resolves through views to the base PII column
audit = spark.sql("""
SELECT user, COUNT(DISTINCT query_id) AS email_query_count
FROM q14_access_history
LATERAL VIEW explode(objects) o AS obj
LATERAL VIEW explode(o.obj.columns) c AS col
WHERE col = 'email'
GROUP BY user
""")
audit.show()
got = {r["user"]: r["email_query_count"] for r in audit.collect()}
assert got == {"analyst_a": 1, "analyst_b": 1}, got     # analyst_b IS caught (view resolved)

# direct-only audit (direct_objects_accessed): the view alias 'contact' hides the access
direct = spark.sql("""
SELECT user, COUNT(DISTINCT query_id) AS n
FROM q14_access_history
LATERAL VIEW explode(direct_objects) o AS obj
LATERAL VIEW explode(o.obj.columns) c AS col
WHERE col = 'email'
GROUP BY user
""")
direct.show()
dg = {r["user"]: r["n"] for r in direct.collect()}
assert dg == {"analyst_a": 1}, dg                        # analyst_b invisible without lineage
print("Q14 OK — objects_accessed audit:", got, "; direct_objects_accessed only:", dg)
print("Q14 note — count DISTINCT queries, not exploded rows (row-level counting over-counts a query")
print("that touches N columns of the same table).")
print("Talk track: objects_accessed resolves through views to base tables (secure views included);")
print("direct_objects_accessed stops at what the query named; ACCOUNT_USAGE lags ~45 min and retains")
print("365 days; QUERY_HISTORY knows tables but NOT columns — that's why ACCESS_HISTORY exists.")

## Q15 · Snowflake — Semi-Structured VARIANT at Scale [HARD]

**Scenario (event platform on Snowflake):** Product events land with varying schemas in a `VARIANT` column. The BI team complains queries are slow and expensive. You must explain the **right** pattern: how micro-partitions prune for variant paths, when to `LATERAL FLATTEN`, and when to **materialize extracted columns** — plus where **Search Optimization** and **Dynamic Tables** fit.

**Requirements:**
1. Write the flatten query (`payload:event_type::string` casting, `LATERAL FLATTEN` over repeated arrays).
2. Explain the cost model: every variant path access reads JSON from the micro-partition column — extracting hot paths into typed `NUMBER/VARCHAR` columns at load/transform time prunes and avoids re-parse.
3. Show the runnable Spark analogue of both patterns (explode-flatten vs materialized typed columns) and prove the outputs agree.
4. Know the accelerators: Search Optimization on variant paths (point lookups), materialized/dynamic tables for repeated flattens, and `ENABLE_UNLOAD_PHYSICAL_TYPE_OPTIMIZATION`-style trivia.

**Interview trap:** flattening a high-cardinality array explodes row counts (storage + micro-partition churn); over-flattening in a view queried by every dashboard is the classic anti-pattern — materialize once instead.

In [0]:
# ---- Reference SQL (Snowflake deliverable; printed) ----
q15_sql = """
-- 1) flatten hot event payloads (the query every dashboard wants)
CREATE OR REPLACE TABLE events_flat AS
SELECT event_id,
       payload:event_type::string   AS event_type,
       payload:user_id::int          AS user_id,
       f.value:amount::decimal(12,2) AS amount
FROM raw_events,
     LATERAL flatten(input => payload:items) f;

-- 2) point lookups on a variant path: search optimization service
ALTER TABLE raw_events ADD SEARCH OPTIMIZATION
  ON EQUALITY(payload:user_id);

-- 3) repeated flatten in a DYNAMIC TABLE (incremental, not per-query)
CREATE OR REPLACE DYNAMIC TABLE events_dt
  TARGET_LAG = '5 minutes'
  WAREHOUSE = 'wh_etl'
AS SELECT event_id, payload:event_type::string AS event_type FROM raw_events;
"""
print(q15_sql)

# ---- runnable Spark analogue: prove flatten == materialized typed columns ----
raw = spark.createDataFrame([
    (1, '{"event_type":"purchase","user_id":42,"items":[{"amount":10.5},{"amount":20.0}]}'),
    (2, '{"event_type":"refund","user_id":7,"items":[{"amount":99.0}]}'),
], "event_id INT, payload STRING")
schema = "event_type STRING, user_id INT, items ARRAY<STRUCT<amount DOUBLE>>"

# pattern A — explode/flatten on read (the per-query cost)
flat = (raw.withColumn("j", F.from_json("payload", schema))
          .select("event_id", "j.event_type", "j.user_id", F.explode("j.items").alias("item"))
          .select("event_id", "event_type", "user_id", "item.amount"))
flat.show()
rows = flat.collect()
assert len(rows) == 3
assert sum(r["amount"] for r in rows) == 129.5
assert {r["event_type"] for r in rows} == {"purchase", "refund"}

# pattern B — materialize typed columns ONCE (the per-row cost, amortized)
mat = (raw.withColumn("j", F.from_json("payload", schema))
          .select("event_id", "j.event_type", "j.user_id",
                  F.aggregate("j.items", F.lit(0.0), lambda acc, x: acc + x.amount).alias("total")))
mat.show()
assert [r["total"] for r in mat.orderBy("event_id").collect()] == [30.5, 99.0]
print("Q15 OK — variant flatten == materialized aggregates; typed extraction wins on hot paths")
print("Talk track: variant path access re-reads JSON from the micro-partition; typed columns prune;")
print("SEARCH OPTIMIZATION adds per-row index cost — only worth it for equality point lookups;")
print("DYNAMIC TABLES replace hand-built materialized flatten pipelines with declared TARGET_LAG.")

## Q16 · Monitoring — Data-Product SLA Contracts [FAANG]

**Scenario (internal data-mesh / product-company SRE):** Each data product publishes an **SLA contract** as code — a row in a registry — and monitoring is *generated from the contract*, not hand-written per table. You design the contract schema and the gate that measures it.

**Requirements:**
1. Contract fields: product, tier (T1 page-on-breach / T2 ticket / T3 digest), **freshness** target (minutes), **volume tolerance** (%), schema-stability policy, owner.
2. Freshness gate: from a run log, compute the latest *successful* run's age vs the contract's freshness target.
3. Volume gate: deviation of the latest run's row count vs the trailing average of successful runs, against the contract tolerance.
4. Tier-based routing: T1 breaches page on-call, T2 open a ticket, T3 lands in the daily digest. Show all three.

**Interview traps:** measuring freshness from the latest run regardless of success state (a failed run is not data — gate on `ok` runs); using a single day's average instead of a trailing window; who gets paged — tiering exists so T3 noise never wakes anyone at 3am.

In [0]:
import datetime as dt
contracts = [
    # product, tier, freshness_min, volume_tolerance_pct, schema_stability, owner
    ("orders_gold",   "T1",   15, 10.0, "locked",        "commerce"),
    ("churn_scores",  "T2", 1440, 20.0, "additive_only", "ml-infra"),
    ("exec_daily",    "T3", 4320, 30.0, "locked",        "bi"),
]
spark.createDataFrame([(p, t, int(f), float(v), s, o) for p, t, f, v, s, o in contracts],
    "product STRING, tier STRING, freshness_min INT, volume_tolerance_pct DOUBLE, schema_stability STRING, owner STRING") \
     .createOrReplaceTempView("q16_contracts")

NOW = dt.datetime(2026, 11, 19, 12, 0, 0)   # the 'now' of the run
runs = [
    # product, run_ts, rows, schema_hash, ok
    ("orders_gold",  "2026-11-19 11:50:00", 100_000, "h1", True),
    ("orders_gold",  "2026-11-19 11:20:00",  99_500, "h1", True),
    ("orders_gold",  "2026-11-18 11:50:00",  50_000, "h1", True),   # yesterday: half volume
    ("churn_scores", "2026-11-19 04:00:00",  12_000, "h2", True),
    ("churn_scores", "2026-11-18 04:00:00",  11_800, "h2", True),
    ("exec_daily",   "2026-11-17 02:00:00",     350, "h3", True),
    ("exec_daily",   "2026-11-16 02:00:00",     320, "h3", False),  # failed run: not data!
]
spark.createDataFrame([(p, dt.datetime.strptime(t, "%Y-%m-%d %H:%M:%S"), int(r), h, ok)
                       for p, t, r, h, ok in runs],
                      "product STRING, run_ts TIMESTAMP, rows BIGINT, schema_hash STRING, ok BOOLEAN") \
     .createOrReplaceTempView("q16_runs")

# freshness gate — latest SUCCESSFUL run age vs contract target
gates = spark.sql("""
WITH latest AS (
  SELECT product, MAX(run_ts) AS last_ok_ts FROM q16_runs WHERE ok GROUP BY product
)
SELECT c.product, c.tier, c.freshness_min, l.last_ok_ts,
       round((unix_timestamp('2026-11-19 12:00:00') - unix_timestamp(l.last_ok_ts)) / 60.0, 0) AS age_min,
       CASE WHEN l.last_ok_ts IS NULL THEN 'NO SUCCESSFUL RUN'
            WHEN (unix_timestamp('2026-11-19 12:00:00') - unix_timestamp(l.last_ok_ts)) / 60.0 > c.freshness_min
            THEN 'FRESHNESS BREACH' ELSE 'fresh' END AS freshness_status
FROM q16_contracts c LEFT JOIN latest l ON l.product = c.product
ORDER BY c.product
""")
gates.show()
g = {r["product"]: r for r in gates.collect()}
last_ok = {"orders_gold": dt.datetime(2026, 11, 19, 11, 50),
           "churn_scores": dt.datetime(2026, 11, 19, 4, 0),
           "exec_daily": dt.datetime(2026, 11, 17, 2, 0)}
sla_map = {c[0]: c[2] for c in contracts}
for p, t in last_ok.items():
    age_min = (NOW - t).total_seconds() / 60.0
    expected = "fresh" if age_min <= sla_map[p] else "FRESHNESS BREACH"
    assert g[p]["freshness_status"] == expected, (p, g[p]["freshness_status"])
    assert abs(g[p]["age_min"] - round(age_min)) <= 1
print("Q16a OK — freshness gate:", {p: g[p]["freshness_status"] for p in g})

# volume gate — latest OK run vs trailing average of OK runs, against contract tolerance
vol = spark.sql("""
WITH base AS (SELECT product, AVG(rows) AS avg_rows FROM q16_runs WHERE ok GROUP BY product),
     last_ok AS (SELECT product, MAX(run_ts) AS last_ok_ts, max_by(rows, run_ts) AS last_rows
                 FROM q16_runs WHERE ok GROUP BY product)
SELECT l.product, l.last_rows, round(b.avg_rows, 1) AS avg_rows,
       round(abs(b.avg_rows - l.last_rows) / b.avg_rows * 100, 2) AS deviation_pct
FROM last_ok l JOIN base b ON l.product = b.product ORDER BY l.product
""")
vol.show()
v = {r["product"]: float(r["deviation_pct"]) for r in vol.collect()}
assert abs(v["orders_gold"] - 20.24) < 0.05 and v["churn_scores"] < 1.0 and v["exec_daily"] == 0.0
tol_map = {c[0]: c[3] for c in contracts}
breach = [p for p, d in v.items() if d > tol_map[p]]
assert breach == ["orders_gold"], breach
print("Q16b OK — volume gate catches orders_gold (20.24% > 10% tolerance); clean:", v)

# tier-based routing: who gets woken up
def route(tier, breach):
    if not breach:
        return "none"
    return {"T1": "page on-call", "T2": "ticket + slack", "T3": "daily digest"}[tier]
assert route("T1", True) == "page on-call" and route("T3", True) == "daily digest"
assert route("T1", False) == "none"
print("Q16 OK — SLA contracts: tiering, freshness gate, volume gate, routing")
print("Talk track: gate on SUCCESSFUL runs only (a failed run is not data); trailing average, not a")
print("single-day baseline; the failed exec_daily run must not contaminate its own baseline;")
print("T3 noise never pages — tiering is the difference between trusted alarms and alert fatigue.")

## Q17 · Monitoring — Test Data Management: Masked Clones + Synthetic Data [FAANG]

**Scenario (enterprise CI/CD for data):** Integration tests need production-shaped data, but regulations forbid PII in lower environments. You design the test-data strategy: **deterministic masked clones** of prod tables plus **synthetic rows** that preserve distributions — and test gates that catch leaks.

**Requirements:**
1. Mask PII deterministically (same input → same token): `email`/`phone` → salted hash tokens, `ssn_last4` → `****`; **join keys must survive masking** so referential tests still pass.
2. Synthetic generation: draw ages/cities/revenues from the observed values so the support and central mass are preserved.
3. A **PII leak test gate**: assert no raw values survive — and prove it works by planting a leak that the gate catches.
4. Joinability: synthetic orders still join the masked customer clone on the original key.

**Interview traps:** nondeterministic masking (random tokens break joins between masked tables); unbalanced synthetic data that breaks cardinality assumptions; and where the salt lives (secret scope, not in the repo) — plus why `ssn_last4` is fully nulled, not tokenized.

In [0]:
import hashlib, random
from pyspark.sql import types as T
random.seed(28)

# prod-like source with PII
prod = [
    (1, "alice@corp.com", "555-0101", "1234", 34, "mumbai", 5000.0),
    (2, "bob@corp.com",   "555-0102", "2345", 45, "delhi",  8200.0),
    (3, "carol@corp.com", "555-0103", "3456", 29, "mumbai", 3200.0),
    (4, "dan@corp.com",   "555-0104", "4567", 51, "pune",   9100.0),
    (5, "erin@corp.com",  "555-0105", "5678", 38, "delhi",  4700.0),
]
pdf = spark.createDataFrame([(i, e, p, s, int(a), c, float(r)) for i, e, p, s, a, c, r in prod],
    "cust_id INT, email STRING, phone STRING, ssn_last4 STRING, age INT, city STRING, revenue DOUBLE")

# masked clone: DETERMINISTIC salted-hash tokens (joins survive), full-null for SSN
def mask(v, salt="d28"):   # production: salt from a secret scope
    return "MASK_" + hashlib.sha256((salt + v).encode()).hexdigest()[:12]
mask_udf = F.udf(mask, T.StringType())
clone = (pdf.withColumn("email", mask_udf("email"))
            .withColumn("phone", mask_udf("phone"))
            .withColumn("ssn_last4", F.lit("****")))
clone.show(truncate=False)
got = {r["cust_id"]: r for r in clone.collect()}
assert got[1]["email"] != prod[0][1] and got[1]["email"].startswith("MASK_")
assert got[1]["ssn_last4"] == "****"
assert mask("alice@corp.com") == mask("alice@corp.com")      # deterministic
assert mask("alice@corp.com") != mask("bob@corp.com")        # distinct
print("Q17a OK — masked clone: PII replaced, deterministic, distinct; SSN fully nulled (not tokenized)")

# synthetic rows: draw from observed values -> support + central mass preserved
ages, revs, cities = [r[4] for r in prod], [r[6] for r in prod], [r[5] for r in prod]
synth = [(100 + i, f"user{i}@test.example.com", f"555-99{i:02d}", "****",
          random.choice(ages), random.choice(cities), float(random.choice(revs)))
         for i in range(50)]
synth_ages = {r[4] for r in synth}; synth_cities = {r[5] for r in synth}
assert synth_ages <= set(ages) and synth_cities <= set(cities)
mean_age = sum(r[4] for r in synth) / 50
assert min(ages) <= mean_age <= max(ages)
print(f"Q17b OK — synthetic rows preserve observed support; mean age {mean_age:.1f} in range")

# PII leak gate: no raw prod values in the clone — and it CATCHES a planted leak
def test_no_raw_pii(df):
    bad = df.filter(F.col("email").endswith("corp.com") |
                    F.col("phone").rlike(r"^555-01\d\d$")).count()
    return bad == 0
assert test_no_raw_pii(clone) is True
poison = clone.withColumn("email", F.lit("leak@corp.com"))    # plant a leak
assert test_no_raw_pii(poison) is False
print("Q17c OK — leak gate passes the clean clone and CATCHES the planted PII leak")

# joinability: orders on the original key still join the masked clone
orders = spark.createDataFrame([(1, 120.0), (3, 80.0), (5, 40.0)], "cust_id INT, amt DOUBLE")
j = orders.join(clone.select("cust_id", "city"), "cust_id")
assert j.count() == 3 and "mumbai" in [r["city"] for r in j.collect()]
print("Q17 OK — masked clone + synthetic data + leak gates + joinability all verified")
print("Talk track: deterministic masking keeps cross-table joins intact (random tokens break them);")
print("synthetic data preserves distributions and cardinality; run the leak gate in CI before any")
print("lower-env table is promoted; SSN is nulled because a token of a 4-digit field is brute-forceable.")

## Q18 · Monitoring — FinOps Cost-Anomaly Attribution (LIVE) [FAANG]

**Scenario (platform FinOps):** Your weekly DBU bill spiked. Finance asks *which day, which SKU, and how much*. This is a **live** exercise on `system.billing.usage` — the system table every FinOps investigation starts from.

**Requirements:**
1. Last 28 days of DBU usage by day; exclude today (partial) from baseline statistics.
2. Find the peak day vs the median — the anomaly definition and its multiplier.
3. **Attribute** the spike: decompose the peak day by SKU against its trailing 7-day average; identify the top contributor and its % increase.
4. State the follow-ups: budgets/alerts (and `record_type` corrections, tag hygiene for per-team attribution).

**Interview traps:** excluding today's partial day from the median (it deflates the baseline and fakes an anomaly every time); `record_type` ORIGINAL/RETRACTION/RESTATEMENT corrections; Decimal sums needing float casts; per-team attribution dying when jobs aren't tagged.

In [0]:
import statistics

# 1) 28-day daily DBU profile (usage_date aggregates faster than usage_start_time)
daily = spark.sql("""
SELECT usage_date, round(sum(usage_quantity), 2) AS dbu
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 28 DAYS AND usage_unit = 'DBU'
GROUP BY usage_date ORDER BY usage_date
""")
daily.show(30)
pdf = daily.toPandas()
vals = [float(x) for x in pdf["dbu"]]                 # Decimal -> float (sum returns Decimal)
assert len(vals) >= 14, len(vals)
med = statistics.median(vals[:-1])                     # EXCLUDE today: partial day fakes anomalies
peak_row = pdf.iloc[:-1]["dbu"].astype(float).idxmax()
peak_date, peak = str(pdf.loc[peak_row, "usage_date"]), float(pdf.loc[peak_row, "dbu"])
print(f"median (excl today) = {med:.2f}; peak day {peak_date} = {peak:.2f} ({peak/max(med,0.01):.1f}x median)")
assert peak > med

# 2) attribute the spike: peak-day DBU by SKU vs its trailing 7-day average
dec = spark.sql(f"""
WITH peak_day AS (
  SELECT sku_name, round(sum(usage_quantity),2) AS peak_dbu
  FROM system.billing.usage
  WHERE usage_date = '{peak_date}' AND usage_unit = 'DBU'
  GROUP BY sku_name
),
base AS (
  SELECT sku_name, round(avg(usage_quantity),2) AS avg_dbu
  FROM system.billing.usage
  WHERE usage_date BETWEEN '{peak_date}' - INTERVAL 7 DAYS AND '{peak_date}' - INTERVAL 1 DAY
    AND usage_unit = 'DBU'
  GROUP BY sku_name
)
SELECT p.sku_name, p.peak_dbu, b.avg_dbu,
       round((p.peak_dbu - b.avg_dbu) / nullif(b.avg_dbu, 0) * 100, 1) AS pct_increase
FROM peak_day p JOIN base b ON p.sku_name = b.sku_name
ORDER BY (p.peak_dbu - b.avg_dbu) DESC
""")
dec.show(10, truncate=False)
rows = dec.collect()
assert len(rows) > 0
top = rows[0]
assert float(top["peak_dbu"]) > 0 and float(top["peak_dbu"]) > float(top["avg_dbu"])
total_spike = sum(float(r["peak_dbu"]) - float(r["avg_dbu"]) for r in rows)
assert total_spike > 0
print(f"Q18 OK — spike {peak_date} ({peak:.2f} DBU, {peak/max(med,0.01):.1f}x median) attributed to")
print(f"      top driver {top['sku_name']} ({float(top['pct_increase'])}% vs 7-day avg)")
print("Follow-ups: budget alert on this SKU (system tables + SQL alerts); check record_type")
print("RETRACTION/RESTATEMENT corrections before reacting; enforce job/cluster tags for per-team")
print("attribution — untagged ALL_PURPOSE compute is where cost investigations go to die.")

# Day 28 Scorecard

Score each question 0-3: **0** = couldn't answer · **1** = needed major hints · **2** = minor gaps · **3** = interview-ready.

| # | Category | Topic | Tag | Score (0-3) | Notes |
| --- | --- | --- | --- | --- | --- |
| 1 | SQL | Sweep-line max concurrency | [FAANG] | | |
| 2 | SQL | RFM NTILE quintiles | [HARD] | | |
| 3 | Python | CRDT PN-counter & LWW set | [FAANG] | | |
| 4 | Python | Skip list | [FAANG] | | |
| 5 | PySpark | Incremental AI enrichment | [FAANG] | | |
| 6 | PySpark | Higher-order functions | [HARD] | | |
| 7 | Spark Opt | Dynamic resource allocation | [HARD] | | |
| 8 | Spark Opt | Lineage & checkpoint truncation | [HARD] | | |
| 9 | ETL | Debezium log CDC | [FAANG] | | |
| 10 | ETL | HMAC PII tokenization | [HARD] | | |
| 11 | ETL | Metadata-driven ingestion | [HARD] | | |
| 12 | Data Modeling | Ratio & non-additive metrics | [HARD] | | |
| 13 | Data Modeling | MRR waterfall | [FAANG] | | |
| 14 | Snowflake | ACCESS_HISTORY forensics | [FAANG] | | |
| 15 | Snowflake | VARIANT at scale | [HARD] | | |
| 16 | Monitoring | SLA contracts | [FAANG] | | |
| 17 | Monitoring | Test data management | [FAANG] | | |
| 18 | Monitoring | FinOps anomaly attribution | [FAANG] | | |

**Target for Week 4:** ≥ 42/54 (2.33 avg), with no question below 2.

**Review rules:** every ⩽ 1 goes on tomorrow's warm-up; re-derive the failed mechanism from scratch before re-attempting.

**Spaced repetition queue (from this week):** Day 25 · lineage checkpoints (Q8 here is the step-up) · Day 26 · AI functions (Q5 is the incremental step-up) · Day 27 · FinOps budgets (Q18 is the attribution step-up).